# TunnelFlow-UB — Ollama on Colab, public in one run

**1.** Runtime → Change runtime type → **T4 GPU** → Save
<img src="https://raw.githubusercontent.com/TheronThayer9999/TunnelFlow-UB/master/docs/img/02-runtime-type-t4-gpu.png" alt="T4 GPU" width="380">

**2.** Connect → press ▶ on the cell below (or **Run all**)

**3.** Copy the last line: `API: http://bore.pub:<PORT>` — that port is already verified with `curl`.

> No GPU? The cell warns and falls back to `qwen2.5:0.5b` on CPU.
> Colab cannot switch the runtime from code and cannot auto-run on Connect — one ▶ is the minimum.


In [ ]:
%%bash
# ==========================================================
# Install Bore + Ollama, open the tunnel, pull the model, test it
# Safe to re-run: it is idempotent.
# ==========================================================
set -e

MODEL="tranhuythang9999/deepseek-v4-pro-9b-heretic:probe"
PORT=11434                 # Ollama port inside Colab
HOST="bore.pub"            # public tunnel server

# 0) GPU? (free T4). No GPU -> small CPU model so this still works.
if nvidia-smi -L > /dev/null 2>&1; then
    echo "GPU: $(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -n 1)"
else
    echo "No GPU -> Runtime > Change runtime type > T4 GPU (free), then Run all again."
    echo "Continuing on CPU with qwen2.5:0.5b."
    MODEL="qwen2.5:0.5b"
fi

# 1) Bore, the tunnel tool
command -v bore > /dev/null || {
    wget -q "https://github.com/ekzhang/bore/releases/download/v0.6.0/bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz"
    tar -xzf bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz
    mv bore /usr/local/bin/ && rm -f bore-v0.6.0-*.tar.gz
}
echo "bore: $(bore --version)"

# 2) Ollama (install.sh often fails at the systemctl step on Colab, so don't abort)
command -v ollama > /dev/null || {
    apt-get update -qq && apt-get install -y -qq zstd curl ca-certificates > /dev/null
    curl -fsSL https://ollama.com/install.sh | sh || true
}

# 3) Serve Ollama on 0.0.0.0, only this tunnel is replaced
pkill -f "ollama serve" || true
pkill -f "bore local $PORT --to" || true
sleep 2
nohup env OLLAMA_HOST=0.0.0.0:$PORT OLLAMA_ORIGINS="*" ollama serve > ollama.log 2>&1 &
for _ in $(seq 1 60); do
    curl -sf "http://127.0.0.1:$PORT/api/tags" > /dev/null && break
    sleep 1
done

# 4) Tunnel + read the port (the digits right after "bore.pub:", never the year in the timestamp)
rm -f bore.log
nohup bore local $PORT --to $HOST > bore.log 2>&1 &
PUB=""
for _ in $(seq 1 30); do
    PUB=$(sed 's/\x1b\[[0-9;]*m//g' bore.log 2>/dev/null \
          | grep -oE "bore\.pub:[0-9]+" | sed 's/.*://' | tail -n 1)
    [ -n "$PUB" ] && break
    sleep 1
done
if [ -z "$PUB" ]; then
    echo "Could not read the port. bore.log:"; tail -n 20 bore.log; exit 1
fi
BASE="http://$HOST:$PUB"
echo "$PUB" > bore.port

# 5) Model + one real call through the public URL
ollama pull "$MODEL"
echo
curl -s --max-time 300 "$BASE/api/generate" \
    -d "{\"model\":\"$MODEL\",\"prompt\":\"Answer in one short sentence: what is 1+1?\",\"stream\":false}" \
    | python3 -c "import sys,json;print('Model says:', json.load(sys.stdin).get('response','').strip())"

echo
echo "API: $BASE"
echo "Test: curl $BASE/api/tags"


In [ ]:
%%bash
# ==========================================================
# OPTIONAL — SSH into Colab. Runs only if the tunnel above is healthy.
# Change the password first.
# ==========================================================
ENABLE_SSH=1
ROOT_PASSWORD="${ROOT_PASSWORD:-MySecurePass123}"

[ "$ENABLE_SSH" = "1" ] || { echo "SSH disabled."; exit 0; }
curl -sf --max-time 10 "http://bore.pub:$(cat bore.port)/api/tags" > /dev/null 2>&1 \
    || { echo "Run the cell above first (its tunnel is not healthy yet)."; exit 0; }

apt-get install -y -qq openssh-server > /dev/null 2>&1 || true
echo "root:${ROOT_PASSWORD}" | chpasswd
mkdir -p /run/sshd
sed -i 's/#\?PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config
sed -i 's/#\?PasswordAuthentication.*/PasswordAuthentication yes/' /etc/ssh/sshd_config
ssh-keygen -A > /dev/null 2>&1
pkill sshd || true
sleep 1
/usr/sbin/sshd

# second tunnel for port 22 (the Ollama tunnel above stays up)
pkill -f "bore local 22 --to" 2>/dev/null || true
rm -f bore_ssh.log
nohup bore local 22 --to bore.pub > bore_ssh.log 2>&1 &
SSH_PORT=""
for _ in $(seq 1 30); do
    SSH_PORT=$(sed 's/\x1b\[[0-9;]*m//g' bore_ssh.log 2>/dev/null \
               | grep -oE "bore\.pub:[0-9]+" | sed 's/.*://' | tail -n 1)
    [ -n "$SSH_PORT" ] && break
    sleep 1
done

echo "ssh -p ${SSH_PORT:-?} root@bore.pub    # password: ${ROOT_PASSWORD}"


In [ ]:
# ==========================================================
# Client sample — same API you would use from your own machine
# ==========================================================
import time
from pathlib import Path

MODEL = "tranhuythang9999/deepseek-v4-pro-9b-heretic:probe"
BASE = "http://bore.pub:" + Path("bore.port").read_text().strip()
print("Endpoint:", BASE)

!pip install -q openai
from openai import OpenAI

client = OpenAI(base_url=BASE + "/v1", api_key="ollama", timeout=300)

t0 = time.time()
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in 5 words."}],
)
print(f"[{time.time() - t0:.1f}s]", r.choices[0].message.content.strip())

# streaming
for chunk in client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Three short lines about the ocean."}],
    stream=True,
):
    print(chunk.choices[0].delta.content or "", end="", flush=True)


## Result of the run above

Old script printed the **year** as the port (`2026`):

<img src="https://raw.githubusercontent.com/TheronThayer9999/TunnelFlow-UB/master/docs/img/01-before-wrong-port-2026.png" alt="wrong port 2026" width="600">

Fixed: `bore.log` really says `bore.pub:9368`, and the public endpoint answers:

<img src="https://raw.githubusercontent.com/TheronThayer9999/TunnelFlow-UB/master/docs/img/03-bore-log-correct-port.png" alt="bore.log 9368" width="700">

<img src="https://raw.githubusercontent.com/TheronThayer9999/TunnelFlow-UB/master/docs/img/04-bore-pub-api-tags.png" alt="curl bore.pub:9368/api/tags" width="700">

From your own machine:

```bash
curl http://bore.pub:<PORT>/api/tags
curl http://bore.pub:<PORT>/api/generate -d '{"model":"tranhuythang9999/deepseek-v4-pro-9b-heretic:probe","prompt":"hi","stream":false}'
```

Files on the Colab VM: `bore.log`, `bore.port`, `ollama.log`.
Stop everything: `pkill ollama; pkill bore`.
